# Chapter 3 Problems

### Notebook setup

In [1]:
# Ensure required Python modules are installed
%pip install --quiet numpy scipy seaborn pandas ministats

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Figures setup
plt.clf()  # needed otherwise `sns.set_theme` doesn't work
sns.set_theme(
    context="paper",
    style="whitegrid",
    palette="colorblind",
    rc={"figure.figsize": (6, 1.6)},
)
%config InlineBackend.figure_format = "retina"

<Figure size 640x480 with 0 Axes>

In [4]:
# Simplified int and float __repr__
np.set_printoptions(legacy='1.25')

In [5]:
# Download datasets/ directory if necessary
from ministats import ensure_datasets
ensure_datasets()

datasets/ directory already exists.


In [6]:
# # Useful colors
# snspal = sns.color_palette()
# blue, orange, purple = snspal[0], snspal[1], snspal[4]
# # red = sns.color_palette("tab10")[3]

# # High-resolution please
# %config InlineBackend.figure_format = 'retina'

$\def\stderr#1{\mathbf{se}_{#1}}$
$\def\stderrhat#1{\hat{\mathbf{se}}_{#1}}$
$\def\Mean{\textbf{mean}}$
$\def\Var{\textbf{var}}$
$\def\Std{\textbf{std}}$
$\def\Freq{\textbf{freq}}$
$\def\RelFreq{\textbf{relfreq}}$
$\def\CI#1{\textbf{CI}_{#1}}$
$\def\CIL#1{\textbf{L}_{#1}}$
$\def\CIU#1{\textbf{U}_{#1}}$
$\def\ci#1{\textbf{ci}_{#1}}$
$\def\cil#1{\textbf{l}_{#1}}$
$\def\ciu#1{\textbf{u}_{#1}}$
$\def\DMeans{\textbf{dmeans}}$
$\def\Prop{\textbf{prop}}$
$\def\DProps{\textbf{dprops}}$
$\def\Err{ {\Large \varepsilon}}$
$\def\calN{\mathcal{N}}$
$\def\Tdist{\mathcal{T}}$
$\def\Expon{\mathrm{Expon}}$

(this cell contains the macro definitions like $\stderr{\overline{\mathbf{x}}}$, $\stderrhat{}$, $\Mean$, ...)

#### Problem NN: alt t-test for the mean of Batch 04 (Example 1BT)

In [7]:
muK0 = 1000   # population mean (expected kombucha volume)

In [8]:
kombucha = pd.read_csv("../datasets/kombucha.csv")
ksample04 = kombucha[kombucha["batch"]==4]["volume"]
n04 = len(ksample04)
obsmean04 = np.mean(ksample04)

In [9]:
# bootstrap estimate for standard error of the mean
from ministats import gen_boot_dist

np.random.seed(42)
kbars_boot04 = gen_boot_dist(ksample04, estfunc=np.mean)
sehat_boot04 = np.std(kbars_boot04)
sehat_boot04

1.225161704465105

In [10]:
# compute the t statistic using bootstrap se
obst04bt = (obsmean04 - muK0) / sehat_boot04
obst04bt

3.1289747190340322

In [11]:
from scipy.stats import t as tdist
from ministats import tailprobs
rvT04 = tdist(n04 - 1)
pvalue04bt = tailprobs(rvT04, obst04bt, alt="two-sided")
pvalue04bt

0.003314349648233729

The $p$-value is very small,
so our decision is to reject $H_0$.

### Problem NN: alt t-test for the mean of Batch 01 (Example 2BT)

In [12]:
muK0 = 1000   # population mean (expected kombucha volume)
sigmaK0 = 10

In [13]:
kombucha = pd.read_csv("../datasets/kombucha.csv")
ksample01 = kombucha[kombucha["batch"]==1]["volume"]
n01 = len(ksample01)
obsmean01 = np.mean(ksample01)

The $p$-value is very large,
so we have no reason to reject $H_0$.

### Problem NN: simulation test of mean

Let's write a reusable function for performing the simulation test for the mean.

In [18]:
## TEST 1 (Do we get the same answer for Batch 04?)
np.random.seed(42)
simulation_test_mean(ksample04, mu0=muK0, sigma0=sigmaK0)

0.0149

In [19]:
## TEST 2 (Do we get the same answer for Batch 01?)
np.random.seed(42)
simulation_test_mean(ksample01, mu0=muK0, sigma0=sigmaK0)

0.5711

We have confirmed the function `simulation_test_mean` works as expected.
We've added it to `ministats` module so we can reuse it later.

### Problem NN: simulation test of var

In [21]:
# Reproduce the results from Example 3
kombucha = pd.read_csv("../datasets/kombucha.csv")
ksample02 = kombucha[kombucha["batch"]==2]["volume"]

np.random.seed(42)
simulation_test_var(ksample02, muK0, sigmaK0, alt="greater")

0.2132

In [22]:
# Reproduce the results from Example 4
kombucha = pd.read_csv("../datasets/kombucha.csv")
ksample08 = kombucha[kombucha["batch"]==8]["volume"]

np.random.seed(43)
simulation_test_var(ksample08, muK0, sigmaK0, alt="greater")

0.0041

We confirm the function `simulation_test_var` gives the same $p$-values as the two samples
we calculated manually above,
so we can be confident it is working as expected.

### P marks improvement

a) Paired t-test
b) Wilcoxon matched-pairs signed-rank

In [31]:
markswide = pd.read_csv("datasets/markswide.csv")
test3 = markswide["test3"]
test1 = markswide["test1"]

ds = test3.values - test1.values
ds

array([-6, 30,  9,  1, -4, 11, 39])